# Imports & Environnement

In [ ]:
import os
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader 
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from IPython.display import Markdown, display

# Initialisation du LLM

In [ ]:
llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0
)

# Chemin vers mon document PDF

In [ ]:
# Je laisse cette partie vide avant de la mettre dans le classroom pour plus de sécurité
pdf_file = "./pdfs/mon_cv.pdf"

# Chargement du PDF

In [ ]:
pdf_loader = PyPDFLoader(pdf_file)

 Découpage (Text Splitter)

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " ", ""]
)

 Application du Splitter et comptage des chunks

In [ ]:
chunks = pdf_loader.load_and_split(text_splitter)
print("Nombre de chunks générés :", len(chunks))

# Modèle d'Embedding Open Source

In [ ]:
embedding_model = OllamaEmbeddings(
    model="all-minilm"
)

 Vector Store (ChromaDB)

In [ ]:
vectorstore = Chroma.from_documents(
    chunks,
    embedding_model,
    collection_name="mon_cv_rag",
    persist_directory="./store"
)

# Définition du Retriever

In [ ]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

Test de recherche de Chunks

In [ ]:
user_input = "Quelles sont les compétences et technologies maîtrisées par le candidat ?"

retrieved_chunks = retriever.invoke(user_input)
print(retrieved_chunks)
print("Nombre de chunks récupérés :", len(retrieved_chunks))

# Prompt Template

In [ ]:
prompt_template = """
Answer the following question based only on provided context.
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!

<context>
{context}
</context>
<question>
{question}
</question>
"""

 Préparation et concaténation du contexte

In [ ]:
relevant_document_chunks = retriever.invoke(user_input)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)

print("Nombre de chunks pertinents :", len(relevant_document_chunks))

 Formatage du prompt

In [ ]:
prompt = prompt_template.format(context=context_for_query, question=user_input)
print(prompt)

# Génération de la réponse par le LLM

In [ ]:
resp = llm.invoke(prompt)
display(Markdown(resp.content))

# Fonction RAG globale

In [ ]:
def RAG(query, llm=llm, prompt_template=prompt_template):
    context_docs = retriever.invoke(query)
    context_list = [d.page_content for d in context_docs]
    context_for_query = ". ".join(context_list)   
    prompt = prompt_template.format(context=context_for_query, question=query)
    resp = llm.invoke(prompt)
    return resp.content

Test anti-hallucination

In [ ]:
resp_hors_sujet = RAG("Quels sont les événements culturels dans le sud-est du Maroc ?")
display(Markdown(resp_hors_sujet))

# Préparation de l'évaluation

In [ ]:
eval_question = "Quelles sont les compétences et technologies maîtrisées par le candidat ?"

relevant_document_chunks = retriever.invoke(eval_question)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)

user_message_template = """
###Question
{question}
###Context
{context}
###Answer
{answer}
"""

answer = RAG(eval_question)
display(Markdown(answer))

LLM Juge (Groundness / Fidélité)

In [ ]:
groundness_rater_system_message = """
Vous êtes chargé d'évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d'IA pour générer la réponse, ainsi qu'une réponse générée par l'IA à la question.
Dans l'entrée, la question commencera par ###Question, le contexte commencera par ###Context, et la réponse générée par L'IA commencera par ###Answer.
Critères d'évaluation :
La tâche consiste à juger dans quelle mesure la réponse respecte la métrique.

1- La métrique n'est pas respectée du tout
2- La métrique n'est respectée que dans une mesure limitée
3- La métrique est respectée dans une bonne mesure
4- La métrique est respectée en grande partie
5- La métrique est entièrement respectée

Métrique :

La réponse doit être dérivée uniquement des informations présentées dans le contexte.

Instructions:

Écrivez d'abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique, en considérant la question et le contexte comme entrées.
Évaluez ensuite dans quelle mesure la métrique est respectée.
Utilisez les informations précédentes pour noter la réponse selon les critères d'évaluation et attribuer un score.
"""

groundness_checker = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

Fonction d'évaluation & Exécution du Juge

In [ ]:
def evaluate(system_message, user_message_template, question, model=groundness_checker):
    retrieved_chunks = retriever.invoke(question)
    context_list = [d.page_content for d in retrieved_chunks]
    context = ". ".join(context_list)
    answer = RAG(question) 
    prompt = f"""
{system_message}

USER:
{user_message_template.format(question=question, context=context, answer=answer)}
"""
    juge_response = model.invoke(prompt)
    return juge_response.content

# Lancement de l'évaluation
verdict = evaluate(groundness_rater_system_message, user_message_template, eval_question)
display(Markdown(verdict))